# Match Simbad Targets with LSSTCam Object Catalogue

For each target star (identified by `simbad_id`, `ra_deg`, `dec_deg`) read from
the input CSV, this notebook:

1. Locates the Butler tract/patch that covers the target coordinates.
2. Loads the corresponding `objectTable_tract` (or equivalent catalogue dataset
   auto-discovered from the registry).
3. Performs a nearest-neighbour sky cross-match using
   `astropy.coordinates.SkyCoord.match_to_catalog_sky`.
4. Retains matches within a configurable search radius.
5. Saves the merged table (targets + matched LSST object columns) to CSV/Parquet.


- reference : https://github.com/lsst/tutorial-notebooks/blob/main/DP1/200_Data_Products/201_Catalogs/201_1_Object_table.ipynb

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-26
- **Last update:** 2026-06-27

## Imports

In [ ]:
import gc
import logging
import os
import re
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
import astropy.units as u

from lsst.daf.butler import Butler
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

## Helper utilities

In [ ]:
def dataset_type_exists(butler, name):
    """Return True if *name* is a registered dataset type in *butler*."""
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

## Logging

In [ ]:
# 1. Récupérer le logger racine (ou créez un logger spécifique: logging.getLogger('mon_code'))
log = logging.getLogger()

# 2. Définir le niveau de log global (DEBUG, INFO, WARNING, ERROR)
log.setLevel(logging.INFO)

# 3. Éviter la duplication des handlers si la cellule est exécutée plusieurs fois
if not log.handlers:
    # 4. Créer un handler qui écrit vers la sortie standard (capturée par Jupyter)
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)

    # 5. Définir le format des messages (heure, niveau, nom du logger, message)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    # 6. Ajouter le handler au logger
    log.addHandler(handler)

# Petit test pour vérifier que ça fonctionne
log.info("Le logging est configuré et fonctionne dans le notebook !")

## Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "MATCHOBJ_02"
DIR_DATA_IN = "data_DEEPCCUTOUTS_01_in"  # reuse the same input directory as NB01
DIR_DATA_OUT = f"data_{NB_TAG}_out"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
log.info(f"Data input : {os.path.abspath(DIR_DATA_IN)}")
log.info(f"Data output: {os.path.abspath(DIR_DATA_OUT)}")
log.info(f"Figs       : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG inside DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

In [ ]:
# ── Butler ────────────────────────────────────────────────────────────────────
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

# ── Input targets ─────────────────────────────────────────────────────────────
target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"

# ── Cross-match search radius ─────────────────────────────────────────────────
MATCH_RADIUS_ARCSEC = 1.2  # maximum separation for a valid match [arcsec]

# ── Band used to retrieve the object table (if band-split tables are used) ────
BANDS = "ugruzy"
BANDSEL = "r"


# build the list of required object columns
OBJ_INFO_COLUMNS = [
    "tract",
    "patch",
    "coord_dec",
    "coord_decErr",
    "coord_ra",
    "coord_raErr",
    "coord_flag",
    "ebv",
]


OBJ_PERBAND_COLUMNS = [
    # aperture flux
    "ap12Flux",
    "ap12FluxErr",
    "ap17Flux",
    "ap17FluxErr",
    "ap25Flux",
    "ap25FluxErr",
    "ap50Flux",
    "ap50FluxErr",
    "psfFlux",
    "psfFluxErr",
    "ra",
    "raErr",
    "dec",
    "decErr",
    "sizeExtendedness",
    "sizeExtendedness_flag",
    "blendedness",
    "invalidPsfFlag",
    "inputCount",
]

OBJ_BANDS_COLUMNS = []
for band in BANDS:
    for flux in OBJ_PERBAND_COLUMNS:
        OBJ_BANDS_COLUMNS.append(f"{band}_{flux}")

OBJ_COLUMNS = OBJ_INFO_COLUMNS + OBJ_BANDS_COLUMNS

log.info("Butler configuration done.")

## Load targets

In [ ]:
df_targets = pd.read_csv(os.path.join(DIR_DATA_IN, target_file))
log.info(f"Loaded {len(df_targets)} targets from {target_file}")
display(df_targets.head())

## Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
log.info(f"Butler initialised | repo: {repo}")

## Auto-discover the object-table dataset type

The catalogue dataset name changed across pipeline versions:

| Pipeline era | Dataset type name          |
|:-------------|:---------------------------|
| Gen2 / HSC   | `deepCoadd_obj`            |
| DP1          | `objectTable_tract`        |
| DP2+         | `object_table_tract`       |

We probe the registry so the notebook is collection-agnostic.


In [ ]:
# Prioritised list of candidate object-table dataset type names
OBJECT_TABLE_CANDIDATES = [
    "object_patch",
    "objectTable",
    "object",
    "objectTable_tract",  # DP1 / recent Science Pipelines
    "object_table_tract",  # DP2+
    "deepCoadd_obj",  # Gen2 / older collections
    "sourceTable_visit",  # visit-level source table (fallback)
]

# List all object-table-related types actually in the registry
all_obj_types = [
    d.name for d in registry.queryDatasetTypes() if "object" in d.name.lower() or "table" in d.name.lower()
]

log.info("Check with notebook 00_QueryButler.ipynb to find availalable objects-datasets ")

# Pick the first candidate that is registered
OBJ_DATASET = None
for name in OBJECT_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        OBJ_DATASET = name
        log.info(f"\n✔ Selected object-table dataset type: '{OBJ_DATASET}'")
        break

if OBJ_DATASET is None:
    raise RuntimeError(
        "No recognised object-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_obj_types}"
    )

## Inspect the schema of the object table (first available tract)

We load a single object table to check which columns are available before
the main loop.

In [ ]:
# Use the coordinates of the first target to locate a representative tract
first_row = df_targets.iloc[0]
probe_point = SpherePoint(first_row["ra_deg"], first_row["dec_deg"], degrees)


probe_tract_info = skymap.findTract(probe_point)
probe_patch_info = probe_tract_info.findPatch(probe_point)

probe_tract_id = probe_tract_info.getId()
probe_patch_id = probe_patch_info.getSequentialIndex()

probe_dataId = {"tract": probe_tract_id, "patch": probe_patch_id, "skymap": skymapName}


log.info(f"Probing schema with tract {probe_tract_id} and patch {probe_patch_id}")
df_probe = butler.get(OBJ_DATASET, dataId=probe_dataId)

# Convert to pandas if the Butler returns an lsst.afw table or an Arrow table
if not isinstance(df_probe, pd.DataFrame):
    try:
        df_probe = df_probe.to_pandas()
    except AttributeError:
        df_probe = pd.DataFrame(df_probe)

log.info("df_probe has %d columns: %s", len(df_probe.columns), list(df_probe.columns))

In [ ]:
# Identify the RA/Dec and objectId column names from the schema
# They can vary between pipeline versions.


def find_col(df, candidates):
    """Return the first column name from *candidates* that exists in *df*."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found in table columns: {list(df.columns[:20])}")


ra_col = find_col(df_probe, ["coord_ra", "ra", "RA", "ra_deg"])
dec_col = find_col(df_probe, ["coord_dec", "dec", "DEC", "dec_deg"])
id_col = find_col(df_probe, ["objectId", "object_id", "id", "sourceId"])

log.info(f"RA column  : {ra_col}")
log.info(f"Dec column : {dec_col}")
log.info(f"ID column  : {id_col}")

## Main cross-match loop

For every target we:
1. Find the tract that contains the target.
2. Load `objectTable_tract` for that tract (cached: each tract is loaded only once).
3. Cross-match with `SkyCoord.match_to_catalog_sky`.
4. Keep the match only if separation ≤ `MATCH_RADIUS_ARCSEC`.


In [ ]:
# Cache: avoid reloading the same patch table multiple times.
# Key: (tract_id, patch_id)  — patch_id is the integer index returned by
# tract_info.findPatch(point).getId().getSequential()
patch_cache: dict[tuple[int, int], pd.DataFrame] = {}


def get_object_table_patch(tract_id: int, patch_id: int) -> pd.DataFrame:
    """Load (and cache) the object table for a single *patch* of *tract_id*.

    Loading by patch instead of the full tract dramatically reduces memory
    consumption on the RSP (a typical DP2 tract has ~1 M objects; a single
    patch holds ~5-20 k objects).
    """
    key = (tract_id, patch_id)
    if key in patch_cache:
        return patch_cache[key]

    # It is mandatory to get the object by tract and Patch otherwise memory is overwhelme.
    dataId = {"tract": tract_id, "patch": patch_id, "skymap": skymapName}
    log.info(f" *** Loading object table for tract {tract_id} / patch {patch_id} … ")

    try:
        df = butler.get(OBJ_DATASET, dataId=dataId)
        if not isinstance(df, pd.DataFrame):
            try:
                df = df.to_pandas()
            except AttributeError:
                df = pd.DataFrame(df)
        log.info(f"\t {len(df):,} objects found tract {tract_id} / patch {patch_id}")
        patch_cache[key] = df
        return df
    except Exception as exc:
        log.info(f"\t >>>>>> FAILED : Exception : {exc}")
        return None


log.info("Patch cache initialised.")

In [ ]:
results = []  # list of dicts; one per target

# loop on simbad targets
for idx, target in df_targets.iterrows():
    simbad_id = target["simbad_id"]
    ra_t = target["ra_deg"]
    dec_t = target["dec_deg"]

    # ── 1.  Find tract **and patch** ─────────────────────────────────────────
    point = SpherePoint(ra_t, dec_t, degrees)
    tract_info = skymap.findTract(point)
    tract_id = tract_info.getId()

    patch_info = tract_info.findPatch(point)
    patch_id = patch_info.sequential_index  # integer patch index

    log.info(
        f"[{idx:3d}] {simbad_id:40s}  ra={ra_t:.5f}  dec={dec_t:+.5f}" f"  tract={tract_id}  patch={patch_id}"
    )

    # ── 2.  Load object table for this patch only (cached) ───────────────────
    df_obj = get_object_table_patch(tract_id, patch_id)

    if df_obj is None or len(df_obj) == 0:
        log.info(f"*** no object table available for tract {tract_id} / patch {patch_id} — skipping")
        row = target.to_dict()
        row.update(
            {
                "lsst_objectId": None,
                "lsst_ra": None,
                "lsst_dec": None,
                "separation_arcsec": None,
                "match_status": "no_table",
                "lsst_tract": tract_id,
                "lsst_patch": patch_id,
            }
        )
        results.append(row)
        continue

    # ── 3.  Build SkyCoord catalogue ─────────────────────────────────────────
    ra_obj = df_obj[ra_col].values
    dec_obj = df_obj[dec_col].values

    if ra_obj.max() <= 2 * np.pi + 0.1:  # heuristic: radians
        unit_sky = u.rad
    else:  # degrees
        unit_sky = u.deg

    cat_sky = SkyCoord(ra=ra_obj * unit_sky, dec=dec_obj * unit_sky)
    tgt_sky = SkyCoord(ra=ra_t * u.deg, dec=dec_t * u.deg)

    # ── 4.  Nearest-neighbour match ──────────────────────────────────────────
    best_idx, sep2d, _ = tgt_sky.match_to_catalog_sky(cat_sky)
    sep_arcsec = sep2d.to(u.arcsec).value[0]

    matched_obj = df_obj.iloc[best_idx]

    row = target.to_dict()
    row["lsst_tract"] = tract_id
    row["lsst_patch"] = patch_id

    if sep_arcsec <= MATCH_RADIUS_ARCSEC:
        lsst_ra = matched_obj[ra_col]
        lsst_dec = matched_obj[dec_col]
        if unit_sky == u.rad:
            lsst_ra = np.degrees(lsst_ra)
            lsst_dec = np.degrees(lsst_dec)

        row.update(
            {
                "lsst_objectId": int(matched_obj[id_col]),
                "lsst_ra": lsst_ra,
                "lsst_dec": lsst_dec,
                "separation_arcsec": sep_arcsec,
                "match_status": "matched",
            }
        )
        msg1 = f" \t matched  objectId={matched_obj[id_col]}"
        msg2 = f" \t sep={sep_arcsec:.3f} arcsec"
        log.info(msg1)
        log.info(msg2)
    else:
        row.update(
            {
                "lsst_objectId": None,
                "lsst_ra": None,
                "lsst_dec": None,
                "separation_arcsec": sep_arcsec,
                "match_status": "no_match",
            }
        )
        log.info(f" \t no match  (closest sep={sep_arcsec:.3f} arcsec > {MATCH_RADIUS_ARCSEC} arcsec)")

    results.append(row)

log.info(f"\nDone: {len(results)} targets processed.")

## Results summary

In [ ]:
df_results = pd.DataFrame(results)

n_total = len(df_results)
n_matched = (df_results["match_status"] == "matched").sum()
n_nomatch = (df_results["match_status"] == "no_match").sum()
n_notable = (df_results["match_status"] == "no_table").sum()

log.info(f"Total targets : {n_total}")
log.info(f"  Matched     : {n_matched}  ({100*n_matched/n_total:.1f}%)")
log.info(f"  No match    : {n_nomatch}  (sep > {MATCH_RADIUS_ARCSEC} arcsec)")
log.info(f"  No table    : {n_notable}  (Butler error)")

display(df_results)

## Diagnostic plots

In [ ]:
df_matched = df_results[df_results["match_status"] == "matched"].copy()

# ── Distribution of separations ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_matched["separation_arcsec"], bins=20, edgecolor="k", linewidth=0.5)
ax.axvline(MATCH_RADIUS_ARCSEC, color="red", linestyle="--", label=f'search radius = {MATCH_RADIUS_ARCSEC}"')
ax.set_xlabel("Separation (arcsec)")
ax.set_ylabel("Number of targets")
ax.set_title("Cross-match separation — Simbad targets vs LSST objects")
ax.legend()
plt.tight_layout()
savefig("crossmatch_separation_histogram")
plt.show()

In [ ]:
# ── RA/Dec offset (target − LSST object) ─────────────────────────────────────
cos_dec = np.cos(np.radians(df_matched["dec_deg"].values))
d_ra = (df_matched["ra_deg"].values - df_matched["lsst_ra"].values) * cos_dec * 3600  # arcsec
d_dec = (df_matched["dec_deg"].values - df_matched["lsst_dec"].values) * 3600  # arcsec

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(d_ra, d_dec, s=20, alpha=0.7)
circle = plt.Circle(
    (0, 0), MATCH_RADIUS_ARCSEC, color="red", fill=False, linestyle="--", label=f'r = {MATCH_RADIUS_ARCSEC}"'
)
ax.add_patch(circle)
ax.axhline(0, color="k", linewidth=0.5)
ax.axvline(0, color="k", linewidth=0.5)
ax.set_xlabel(r"$\Delta\,\alpha\cos\delta$ (arcsec)")
ax.set_ylabel(r"$\Delta\,\delta$ (arcsec)")
ax.set_title("Positional offsets: Simbad − LSST")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
savefig("crossmatch_radec_offsets")
plt.show()

In [ ]:
# ── Summary pie chart ─────────────────────────────────────────────────────────
counts = df_results["match_status"].value_counts()
fig, ax = plt.subplots(figsize=(4, 4))
ax.pie(counts.values, labels=counts.index, autopct="%1.0f%%", startangle=90)
ax.set_title(f'Match status (r < {MATCH_RADIUS_ARCSEC}" )')
plt.tight_layout()
savefig("crossmatch_status_pie")
plt.show()

## Save results

In [ ]:
out_stem = os.path.join(DIR_DATA_OUT, "targets_matched_lsst_objects")

df_results.to_csv(out_stem + ".csv", index=False)
df_results.to_parquet(out_stem + ".parquet", index=False)

print(f"Full results  → {out_stem}.csv  /  .parquet")
print(f"  {len(df_results)} rows total, {n_matched} matched")

# Save the matched-only subset for downstream notebooks
out_matched = os.path.join(DIR_DATA_OUT, "targets_matched_only")
df_matched.to_csv(out_matched + ".csv", index=False)
df_matched.to_parquet(out_matched + ".parquet", index=False)
print(f"Matched only  → {out_matched}.csv  /  .parquet")